# 03 · Prepare HRNet keypoints
Inspect the MSKA keypoint release, normalize (shoulder-centered, train-only
stats), add velocities, dump `[T, 79·5]` fp16 per id. **Gate:** shoulder
indices set, `normalization_stats.json` saved, counts match RGB.

In [ ]:
# --- Colab bootstrap (run first in every notebook) ---
from google.colab import drive
drive.mount('/content/drive')

import sys
PROJECT = '/content/drive/MyDrive/cslr_phoenix/project'   # <- where this code lives
sys.path.append(PROJECT)
%cd $PROJECT

!pip -q install pyyaml
from src.utils import load_config
cfg = load_config('config.yaml')
print('config loaded:', cfg['project']['name'])


## ADAPT_HERE #3 — inspect raw format & map ids

In [ ]:
import numpy as np, pickle, glob, os
RAW = cfg['paths']['hrnet_keypoints_raw']
print('contents:', os.listdir(RAW)[:20])
# MSKA typically ships a pickle keyed by sample id -> array [T, num_points, 2or3].
# Load and inspect ONE entry, then implement get_raw(split, sid) accordingly.
# Example (adjust to real structure):
# data = pickle.load(open(os.path.join(RAW, 'phoenix2014_keypoints.pkl'),'rb'))
# k = next(iter(data)); print(k, np.array(data[k]).shape)

In [ ]:
# Define this to return [T, num_points, C>=2] for a sample id:   ADAPT_HERE
def get_raw(split, sid):
    raise NotImplementedError('map MSKA structure -> array [T, num_points, C]')

# Sanity: plot one frame's skeleton to read off shoulder indices.
import matplotlib.pyplot as plt
# kp = get_raw('train', SOME_ID)[0]; plt.scatter(kp[:,0], -kp[:,1])
# for i,(x,y) in enumerate(kp[:, :2]): plt.annotate(str(i), (x,-y), fontsize=6)
# plt.gca().set_aspect('equal'); plt.show()
# -> set keypoints.left_shoulder_idx / right_shoulder_idx in config.yaml

## Normalize + velocity features

In [ ]:
import numpy as np
kp_cfg = cfg['keypoints']
LS, RS = kp_cfg['left_shoulder_idx'], kp_cfg['right_shoulder_idx']
assert LS >= 0 and RS >= 0, 'set shoulder indices in config.yaml first (NB cell above)'

def featurize(arr):                       # arr: [T, P, C>=2]
    xy = arr[..., :2].astype(np.float32)
    conf = (arr[..., 2:3] if arr.shape[-1] > 2
            else np.ones((*arr.shape[:2], 1), np.float32)).astype(np.float32)
    mid = ((xy[:, LS] + xy[:, RS]) / 2.0)            # [T,2]
    width = np.linalg.norm(xy[:, LS] - xy[:, RS], axis=-1, keepdims=True)  # [T,1]
    scale = np.median(width[width > 1e-3]) if np.any(width > 1e-3) else 1.0
    center = np.median(mid, axis=0)                  # per-video, stable
    xy = (xy - center[None, None]) / (scale + 1e-6)
    vel = np.zeros_like(xy); vel[1:] = xy[1:] - xy[:-1]
    feat = np.concatenate([xy, conf, vel], axis=-1)  # [T,P,5] -> x,y,conf,dx,dy
    return feat.reshape(feat.shape[0], -1).astype(np.float32)

## Train-only standardization stats, then dump all splits

In [ ]:
from src.utils import load_json, p, save_json
import numpy as np

# Pass 1: accumulate mean/std over TRAIN ONLY.
train_items = load_json(p(cfg, 'manifests') / 'train.json')
s, ss, cnt = None, None, 0
for it in train_items:
    f = featurize(get_raw('train', it['id']))
    s  = f.sum(0) if s  is None else s  + f.sum(0)
    ss = (f**2).sum(0) if ss is None else ss + (f**2).sum(0)
    cnt += len(f)
mean = s / cnt
std = np.sqrt(np.maximum(ss / cnt - mean**2, 1e-8))
save_json({'mean': mean.tolist(), 'std': std.tolist()},
          p(cfg, 'features_kp') / 'normalization_stats.json')
print('train-only stats saved; dim =', len(mean))

In [ ]:
# Pass 2: standardize + dump every split with the SAME stats.
for split in cfg['dataset']['splits']:
    items = load_json(p(cfg, 'manifests') / f'{split}.json')
    out_dir = p(cfg, 'features_kp') / split; out_dir.mkdir(parents=True, exist_ok=True)
    done = {f.stem for f in out_dir.glob('*.npy')}
    for n, it in enumerate(items):
        if it['id'] in done:
            continue
        f = (featurize(get_raw(split, it['id'])) - mean) / std
        np.save(out_dir / f"{it['id']}.npy", f.astype('float16'))
        if n % 200 == 0:
            print(f'  {split} {n}/{len(items)}')
    print(f'{split}: {len(list(out_dir.glob("*.npy")))}/{len(items)} dumped')